In [6]:
import scipy
import logging
import pickle
import datetime
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
SUBJECT_IDS = [f'S{i}' for i in range(2, 12)] + [f'S{i}' for i in range(13, 18)]  # S2 to S11 and S13 to S17
TEST_SUBJECTS = 5
SAMPLING_RATE = 4
SAMPLING_RATES = {
    'chest': {
        'ACC': 700,
        'ECG': 700,
        'EMG': 700,
        'EDA': 700,
        'Resp': 700,
        'Temp': 700
    },
    'wrist': {
        'ACC': 32,
        'BVP': 64,
        'EDA': 4,
        'TEMP': 4
    },
    'label': 700
}

In [7]:
# create a vector from the data frame (signal imported by pandas)
def create_df_array(dataframe):
    matrix_df=dataframe.values
    # returns 2-d matrix
    matrix = np.array(matrix_df)
    array_df = matrix.flatten()# Convert matrix into an array
    return array_df

# convert UTC arrays to arrays in seconds relative to 0 (record beginning)
def time_abs_(UTC_array):
    new_array=[]
    for utc in UTC_array:
        time=(datetime.datetime.strptime(utc,'%Y-%m-%d %H:%M:%S')-datetime.datetime.strptime(UTC_array[0], '%Y-%m-%d %H:%M:%S')).total_seconds()
        new_array.append(int(time))
    return new_array

In [24]:
def read_signals(main_folder):
    signal_dict = {}
    time_dict = {}
    fs_dict = {}

    # Get a list of subfolders in the main folder
    subfolders = [p for p in Path(main_folder).iterdir() if p.is_dir()]

    utc_start_dict={}
    for folder_path in subfolders:
            csv_path = folder_path / 'EDA.csv'
            df=pd.read_csv(csv_path)
            utc_start_dict[folder_path.name]= df.columns.tolist()

    # Iterate over the subfolders
    for folder_path in subfolders:
        folder_name = folder_path.name

        # Initialize a dictionary for the signals in the current subfolder
        signals = {}
        time_line = {}
        fs_signal= {}
        
        # Define the list of desired file names
        desired_files = ['EDA.csv', 'BVP.csv', 'TEMP.csv', 'tags.csv', 'ACC.csv']
   
        # Iterate over the files in the subfolder
        for file_path in folder_path.iterdir():
            file_name = file_path.name

            # Check if it's a CSV file and if it is in the desired files list
            if file_name.endswith('.csv') and file_name in desired_files:
                # Read the CSV file and store the signal data

                if file_name == 'tags.csv':
                    try:
                        df = pd.read_csv(file_path, header=None)
                        tags_vector = create_df_array(df)
                        tags_UTC_vector = np.insert(tags_vector, 0, utc_start_dict[folder_name])
                        signal_array = time_abs_(tags_UTC_vector)
                    except pd.errors.EmptyDataError:
                        signal_array=[]
                else:
                    df = pd.read_csv(file_path)
                    fs = df.loc[0, df.columns[0]]
                    fs = int(fs)  # Get sampling frequency
                    df.drop([0], axis=0, inplace=True)
                    signal_array = df.values
                    time_array = np.linspace(0, len(signal_array) / fs, len(signal_array))
                
                signal_name = file_name.split('.')[0]
                signals[signal_name] = signal_array
                time_line[signal_name] = time_array
                fs_signal[signal_name] = fs

        # Store the signals of the current subfolder in the main dictionary
        signal_dict[folder_name] = signals
        time_dict[folder_name] = time_line
        fs_dict[folder_name] = fs_signal

    return signal_dict, time_dict, fs_dict

In [25]:
data_path = Path('physionet.org') / "files" / "wearable-device-dataset" / "1.0.1" / "Wearable_Dataset" / "STRESS"
participants = list(data_path.iterdir())
signal_data, time_data, fs_dict = read_signals(data_path) # Returns three dictionaries with subjects info: raw signals (signal_data), temporal data ready to graph (time_data) and sample frequency for escha signal(fs_dict).

In [26]:
fs_dict

{'S05': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'S16': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'f12': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'f18': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'f02': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'S02': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'f16': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'S10': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'f11': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'S04': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'S01': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'S18': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'f06': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'f07': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'f03': {'ACC': 32, 'TEMP': 4, 'tags': 4, 'EDA': 4, 'BVP': 64},
 'f14_a': {'ACC': 32, 'TEMP': 4, 'tags':

In [27]:
signal_data['S05'].keys()

dict_keys(['ACC', 'TEMP', 'tags', 'EDA', 'BVP'])

In [51]:
for subject in sorted(signal_data.keys()):
    if subject.startswith('S'):
        print(subject, end=': ')
        print(signal_data[subject]['tags'][2] - signal_data[subject]['tags'][1], signal_data[subject]['tags'][1])
    else:
        print(subject, end=': ')
        print(signal_data[subject]['tags'][1])

S01: 247 371
S02: 192 1
S03: 187 278
S04: 180 181
S05: 183 196
S06: 193 55
S07: 163 115
S08: 176 44
S09: 192 154
S10: 184 67
S11: 170 24
S12: 199 105
S13: 209 173
S14: 210 202
S15: 207 198
S16: 193 330
S17: 174 302
S18: 184 237
f01: 287
f02: 851
f03: 785
f04: 1009
f05: 2172
f06: 587
f07: 783
f08: 523
f09: 745
f10: 229
f11: 720
f12: 2337
f13: 1094
f14_a: 

IndexError: list index out of range